In [19]:
import os
os.environ["GIT_PYTHON_REFRESH"] = "quiet"

import sys
import time
import math
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="gymnasium")

import numpy as np              # type: ignore
import pandas as pd             # type: ignore
import matplotlib.pyplot as plt # type: ignore
print(f"Python: {sys.version}\n")

import mujoco                   # type: ignore
import myosuite                 # type: ignore
import gymnasium as gym         # type: ignore
import stable_baselines3        # type: ignore

import torch                    # pyright: ignore[reportMissingImports]
import torch.nn as nn           # pyright: ignore[reportMissingImports]
import torch.optim as optim     # pyright: ignore[reportMissingImports]

print("MuJoCo:", mujoco.__version__)
print("Myosuite:", myosuite.__version__)
print("Gymnasium:", gym.__version__)
print("Stable Baselines3:", stable_baselines3.__version__)
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.is_available())

Python: 3.10.8 (tags/v3.10.8:aaaf517, Oct 11 2022, 16:50:30) [MSC v.1933 64 bit (AMD64)]

MuJoCo: 3.3.0
Myosuite: 2.11.6
Gymnasium: 0.29.1
Stable Baselines3: 2.7.1
PyTorch: 2.10.0+cpu
GPU: False


In [20]:
all_envs = gym.envs.registry.keys()
myo_envs = sorted([e for e in all_envs if 'myo' in e.lower()])

print(f"Total MyoSuite envs: {len(myo_envs)} | All environments: {len(all_envs)}\n")
print(list(all_envs)), print(myo_envs)

Total MyoSuite envs: 394 | All environments: 449

['CartPole-v0', 'CartPole-v1', 'MountainCar-v0', 'MountainCarContinuous-v0', 'Pendulum-v1', 'Acrobot-v1', 'phys2d/CartPole-v0', 'phys2d/CartPole-v1', 'phys2d/Pendulum-v0', 'LunarLander-v2', 'LunarLanderContinuous-v2', 'BipedalWalker-v3', 'BipedalWalkerHardcore-v3', 'CarRacing-v2', 'Blackjack-v1', 'FrozenLake-v1', 'FrozenLake8x8-v1', 'CliffWalking-v0', 'Taxi-v3', 'tabular/Blackjack-v0', 'tabular/CliffWalking-v0', 'Reacher-v2', 'Reacher-v4', 'Pusher-v2', 'Pusher-v4', 'InvertedPendulum-v2', 'InvertedPendulum-v4', 'InvertedDoublePendulum-v2', 'InvertedDoublePendulum-v4', 'HalfCheetah-v2', 'HalfCheetah-v3', 'HalfCheetah-v4', 'Hopper-v2', 'Hopper-v3', 'Hopper-v4', 'Swimmer-v2', 'Swimmer-v3', 'Swimmer-v4', 'Walker2d-v2', 'Walker2d-v3', 'Walker2d-v4', 'Ant-v2', 'Ant-v3', 'Ant-v4', 'Humanoid-v2', 'Humanoid-v3', 'Humanoid-v4', 'HumanoidStandup-v2', 'HumanoidStandup-v4', 'GymV21Environment-v0', 'GymV26Environment-v0', 'motorFingerReachFixed-v0', '

(None, None)

In [21]:
env = gym.make("CartPole-v0")

print(f"Observation space: {env.observation_space}")
print(f"Action space:      {env.action_space}")

obs, _ = env.reset()
env.render()

print(f"\nObservation shape: {obs.shape}")
print(f"First 6 values:    {obs[:6].round(4)}")

env.close()

Observation space: Box([-4.8000e+00 -3.4028e+38 -4.1888e-01 -3.4028e+38], [4.8000e+00 3.4028e+38 4.1888e-01 3.4028e+38], (4,), float32)
Action space:      Discrete(2)

Observation shape: (4,)
First 6 values:    [ 0.0466 -0.0026 -0.011  -0.0098]


z:\SIM\.venv\lib\site-packages\gymnasium\envs\registration.py:513: DeprecationWarning: WARN: The environment CartPole-v0 is out of date. You should consider upgrading to version `v1`.
  logger.deprecation(


## MyoSuite

In [25]:
# run this before you create / use any env that may call the renderer
import myosuite.renderer.mj_renderer as mjr
import warnings

def _safe_close(self):
    """Close renderer window safely without calling interactive quit()."""
    try:
        # close the GLFW/window object if present
        win = getattr(self, "_window", None)
        if win is not None:
            try:
                win.close()
            except Exception:
                # Some viewer implementations expose different close methods
                try:
                    win.destroy()
                except Exception:
                    pass
        # unset references
        self._window = None
        self._user_exit = True
    except Exception as e:
        warnings.warn(f"Safe close failed: {e}")

# apply monkeypatch
mjr.MJRenderer.close = _safe_close

print("Patched MJRenderer.close() — it will no longer call quit().")

Patched MJRenderer.close() — it will no longer call quit().


In [26]:
# MyoSuite Predefined ENVs

# env = gym.make('myoElbowPose1D6MRandom-v0')
# env = gym.make('myoElbowPose1D6MExoRandom-v0')
env = gym.make('myoArmReachRandom-v0')

env.reset()
env.mj_render()

print(f"Observation space: {env.observation_space.shape}")
print(f"Action space: {env.action_space.shape}")

for _ in range(1000):
    action = env.action_space.sample()
    obs, reward, _, done, info = env.step(action)

    if done: env.reset()
            
env.close()

obs, reward, _, done, info

Observation space: (78,)
Action space: (32,)


(array([-1.9077e-01,  8.1572e-02, -8.1602e-02,  1.9075e-01, -3.9443e-02,
         3.1767e-01,  1.3857e-01, -1.3847e-01, -3.1777e-01,  3.9599e-02,
         1.7459e+00,  7.9371e-01, -1.7461e+00,  1.4058e-01,  2.2846e-02,
         7.2113e-02,  2.2416e-01,  7.9406e-01, -1.2231e-01, -9.3342e-02,
         3.0278e-03, -1.0002e-03,  1.0040e-03, -3.0161e-03,  4.1191e-04,
        -2.2939e-03, -5.7455e-03,  5.6971e-03,  2.2864e-03, -3.8847e-04,
        -2.2357e-03, -7.8019e-03,  2.1561e-03, -1.2014e-01,  2.9231e-02,
        -5.9583e-03,  1.2635e-03, -2.2465e-03, -4.9554e-02,  3.3867e-03,
         1.9655e-02, -4.0088e-01,  1.0115e+00, -2.3740e-01,  1.9760e-01,
         6.9439e-01,  4.9956e-01,  3.3949e-01,  3.9153e-01,  4.9205e-01,
         5.4514e-01,  1.5679e-01,  4.7539e-01,  5.0117e-01,  3.0836e-01,
         1.3020e-01,  8.3853e-02,  3.5657e-01,  1.5900e-01,  4.5107e-01,
         5.3330e-01,  6.3619e-01,  1.0670e-01,  2.5293e-01,  1.3825e-01,
         2.8269e-01,  1.8662e-01,  7.2689e-01,  8.0

## Ref